In [2]:
class Vector:
    def __init__(self, data):
        self.data = list(data)
        self.size = len(self.data)

    def __repr__(self):
        return f"Vector({self.data})"

    def __add__(self, other):
        return Vector([a + b for a, b in zip(self.data, other.data)])

    def __sub__(self, other):
        return Vector([a - b for a, b in zip(self.data, other.data)])

    def __mul__(self, scalar):
        return Vector([x * scalar for x in self.data])

    def dot(self, other):
        return sum(a * b for a, b in zip(self.data, other.data))

    def magnitude(self):
        return sum(x**2 for x in self.data) ** 0.5

In [1]:
class Matrix:
    def __init__(self, data):
        self.data = [list(row) for row in data]
        self.rows = len(self.data)
        self.cols = len(self.data[0])
        self.shape = (self.rows, self.cols)

    def __repr__(self):
        rows_str = "\n  ".join(str(row) for row in self.data)
        return f"Matrix({self.shape}):\n  {rows_str}"

    def __add__(self, other):
        return Matrix(
            [
                [self.data[i][j] + other.data[i][j] for j in range(self.cols)]
                for i in range(self.rows)
            ]
        )

    def __sub__(self, other):
        return Matrix(
            [
                [self.data[i][j] - other.data[i][j] for j in range(self.cols)]
                for i in range(self.rows)
            ]
        )

    def scalar_multiply(self, scalar):
        return Matrix(
            [[self.data[i][j] * scalar for j in range(self.cols)] for i in range(self.rows)]
        )

    def element_wise_multiply(self, other):
        return Matrix(
            [
                [self.data[i][j] * other.data[i][j] for j in range(self.cols)]
                for i in range(self.rows)
            ]
        )

    def matmul(self, other):
        return Matrix(
            [
                [
                    sum(self.data[i][k] * other.data[k][j] for k in range(self.cols))
                    for j in range(other.cols)
                ]
                for i in range(self.rows)
            ]
        )

    def transpose(self):
        return Matrix([[self.data[j][i] for j in range(self.rows)] for i in range(self.cols)])

    def determinant(self):
        if self.shape == (1, 1):
            return self.data[0][0]
        if self.shape == (2, 2):
            return self.data[0][0] * self.data[1][1] - self.data[0][1] * self.data[1][0]
        det = 0
        for j in range(self.cols):
            minor = Matrix(
                [[self.data[i][k] for k in range(self.cols) if k != j] for i in range(1, self.rows)]
            )
            det += ((-1) ** j) * self.data[0][j] * minor.determinant()
        return det

    def inverse_2x2(self):
        det = self.determinant()
        if det == 0:
            raise ValueError("Matrix is singular, no inverse exists")
        return Matrix(
            [
                [self.data[1][1] / det, -self.data[0][1] / det],
                [-self.data[1][0] / det, self.data[0][0] / det],
            ]
        )

    @staticmethod
    def identity(n):
        return Matrix([[1 if i == j else 0 for j in range(n)] for i in range(n)])

In [3]:
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])

print("A + B =", (A + B).data)
print("A @ B =", A.matmul(B).data)
print("A^T =", A.transpose().data)
print("det(A) =", A.determinant())
print("A^-1 =", A.inverse_2x2().data)

I = Matrix.identity(2)
print("A @ A^-1 =", A.matmul(A.inverse_2x2()).data)

A + B = [[6, 8], [10, 12]]
A @ B = [[19, 22], [43, 50]]
A^T = [[1, 3], [2, 4]]
det(A) = -2
A^-1 = [[-2.0, 1.0], [1.5, -0.5]]
A @ A^-1 = [[1.0, 0.0], [0.0, 1.0]]


In [4]:
import random

inputs = Matrix([[0.5], [0.8], [0.2]])
weights = Matrix([[random.uniform(-1, 1) for _ in range(3)] for _ in range(2)])
bias = Matrix([[0.1], [0.1]])


def relu_matrix(m):
    return Matrix([[max(0, val) for val in row] for row in m.data])


pre_activation = weights.matmul(inputs) + bias
output = relu_matrix(pre_activation)

print(f"Input shape: {inputs.shape}")
print(f"Weight shape: {weights.shape}")
print(f"Output shape: {output.shape}")
print(f"Output: {output.data}")

Input shape: (3, 1)
Weight shape: (2, 3)
Output shape: (2, 1)
Output: [[0.3392761368203145], [0.6332727671065876]]



import shutil
shutil.make_archive('v4_download', 'zip', '.', 'v4')
print("Done! v4_download.zip created.")


In [2]:
import torch

ckpt = torch.load("checkpoints/final.pt", map_location="cpu")
print(type(ckpt))

# if dict
if isinstance(ckpt, dict):
    for k, v in ckpt.items():
        shape = v.shape if hasattr(v, "shape") else type(v)
        print(f"  {k}: {shape}")
else:
    print(ckpt)

<class 'dict'>
  epoch: <class 'int'>
  G_state: <class 'collections.OrderedDict'>
  G_state_ema: <class 'collections.OrderedDict'>
  D_state: <class 'collections.OrderedDict'>
  opt_G_state: <class 'dict'>
  opt_D_state: <class 'dict'>
  config: <class 'dict'>


In [3]:
# Check what epoch training stopped at
print(ckpt["epoch"])

# Check Generator architecture (layer names + shapes)
for k, v in ckpt["G_state"].items():
    print(k, v.shape)

# Check saved config
import json

print(json.dumps(ckpt["config"], indent=2))

350
district_embed.weight torch.Size([71, 16])
state_embed.weight torch.Size([36, 8])
fc.0.weight torch.Size([6144, 234])
fc.0.bias torch.Size([6144])
up1.conv_t.weight torch.Size([512, 256, 4, 1])
up1.cbn.bn.running_mean torch.Size([256])
up1.cbn.bn.running_var torch.Size([256])
up1.cbn.bn.num_batches_tracked torch.Size([])
up1.cbn.affine.weight torch.Size([512, 106])
up1.cbn.affine.bias torch.Size([512])
up1.shortcut.1.weight torch.Size([256, 512, 1, 1])
up1.shortcut.2.bn.running_mean torch.Size([256])
up1.shortcut.2.bn.running_var torch.Size([256])
up1.shortcut.2.bn.num_batches_tracked torch.Size([])
up1.shortcut.2.affine.weight torch.Size([512, 106])
up1.shortcut.2.affine.bias torch.Size([512])
attn1.gamma torch.Size([1])
attn1.query.weight torch.Size([32, 256, 1, 1])
attn1.query.bias torch.Size([32])
attn1.key.weight torch.Size([32, 256, 1, 1])
attn1.key.bias torch.Size([32])
attn1.value.weight torch.Size([256, 256, 1, 1])
attn1.value.bias torch.Size([256])
up2.conv_t.weight torch